In [1]:
#spark session
from pyspark.sql import SparkSession
# this means spark://location of the spark master node:7077 is the port number of the spark master node
spark = SparkSession.builder.appName("Cluster Execution").master("spark://1d82a0b4d4fd:7077").getOrCreate()
spark

In [2]:
# Create a sample data frame
df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [3]:

df.write.format("csv").option("header", True).mode("overwrite").save("/data/output/15/2/range.csv")
# in spark ui if we go to http://localhost:4040/jobs/
# in the Completed Jobs we can see save action is completed with 40 tasks.
# This means that the data frame is divided into 40 partitions and each partition is written to a separate file in the output directory.

# so this purticular task has been executed on the cluster.

In [9]:
# now we will add 2 executer each in 1 node for both with 4 cores and 512 MB memory
from pyspark.sql import SparkSession
# 2 executors, 4 cores and 512 MB each
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("Cluster Execution")
         .master("spark://1d82a0b4d4fd:7077")
         .config("spark.executor.instances", 4) # number of executors
         .config("spark.executor.cores", 4)  # cores per executor
         .config("spark.executor.memory", "512m")  # heap per executor
         .getOrCreate())

# this all are happening in the client mode,when we check on the http://localhost:4040/executors/
# we can see the driver is running on address 0a8b05ba503f:39005 and all others are running on 172.18.0.x:xxxx worker nodes,
# that means driver is running from local machine 

In [10]:
# Create a sample data frame
df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [11]:
df.write.format("csv").option("header", True).mode("overwrite").save("/data/output/15/2/range.csv")

In [ ]:
# this all are happening in the client mode,when we check on the http://localhost:4040/executors/
# we can see the driver is running on address 0a8b05ba503f:39005 and all others are running on 172.18.0.x:xxxx worker nodes,
# that means driver is running from local machine 

# because of this the best way to run in cluster mode is spark-submit command.

# the script is chaper11_understandCluster.py - it has NO .master(), spark-submit supplies it.
# docker-compose.yml now mounts this whole chapter folder into the jupyter container:
#   ../  ->  /home/jupyter/chapter1
# so the script can stay where it is (no need to move it into data/) and is still
# visible inside the container. apply the change once with:
#   docker compose -f docker-images/docker-compose.yml up -d
# (only the jupyter container is recreated; master + workers keep running)

# get the CWD to spark installation folder
#   docker exec -it bd-pyspark-jupyter-lab bash
#   cd /spark

# now submit it:
#   ./bin/spark-submit \
#     --master spark://1d82a0b4d4fd:7077 \
#     --total-executor-cores 16 \
#     --executor-cores 4 \
#     --executor-memory 512M \
#     /home/jupyter/chapter1/chaper11_understandCluster.py

# --total-executor-cores 16 / --executor-cores 4 = 4 executors (16 / 4)
# --num-executors is IGNORED in standalone, same as spark.executor.instances
# --master: 1d82a0b4d4fd is the master container id, so it changes if that container is
#           recreated; spark://bd-spark-master:7077 is the compose service name and never changes
# the script writes to /data (the spark_data volume) so master + both workers can reach the path
# watch it at http://localhost:8080 - the app shows up while it runs and moves to
# Completed Applications when the script exits (a notebook session stays RUNNING until spark.stop())

In [8]:
# Stop Spark Section
spark.stop()